# Refresh `model_benchmarks` (Artificial Analysis + LLM Stats)

**Daily** job: always **overwrites** `model_benchmarks` (current snapshot). The history table `model_benchmarks_history` is **appended only when it's been ≥ 14 days** since the last snapshot — so a single daily schedule self-paces the biweekly history cadence.

Reuses the merge helpers from `model-cap-benchmarks/build_models_json.py`. Keys come from the `frontier_labs` secret scope.

**Prereq:** secrets `AA_API_KEY` (aa_… key) + `LLM_STATS_KEY` (sk_… key) in scope `frontier_labs`.

In [ ]:
# --- config ---
CATALOG = "fso_market_intelligence"
SCHEMA  = "frontier_labs"
SCOPE   = "frontier_labs"
AA_SECRET = "AA_API_KEY"
LS_SECRET = "LLM_STATS_KEY"
HISTORY_EVERY_DAYS = 14   # append to history only if this many days since last snapshot

# Auto-locate model-cap-benchmarks in the workspace (so the Git-sourced job needs no path edits)
import subprocess, os
_hits = subprocess.run(["find", "/Workspace", "-maxdepth", "8", "-name", "build_models_json.py"],
                       capture_output=True, text=True).stdout.strip().splitlines()
assert _hits, "build_models_json.py not found under /Workspace — pull the Git folder first"
MCB = os.path.dirname(_hits[0])
print("MCB =", MCB)

In [ ]:
# Reuse the tested merge helpers from the repo (import is side-effect free — main() is guarded)
import sys
if MCB not in sys.path:
    sys.path.insert(0, MCB)
from build_models_json import build_ls_lookup, enrich_from_ls, org_country

In [ ]:
# --- fetch (keys from secrets) ---
import requests, time

AA_KEY = dbutils.secrets.get(SCOPE, AA_SECRET)
LS_KEY = dbutils.secrets.get(SCOPE, LS_SECRET)

def aa_get(path, params=None):
    for a in range(4):
        r = requests.get("https://artificialanalysis.ai/api/v2" + path,
                         headers={"x-api-key": AA_KEY, "Accept": "application/json"},
                         params=params, timeout=60)
        if r.status_code == 429:
            time.sleep(15 * (a + 1)); continue
        r.raise_for_status(); return r.json()
    r.raise_for_status()

def ls_get(path, params=None):
    for a in range(4):
        try:
            r = requests.get("https://api.llm-stats.com/stats/v1" + path,
                             headers={"Authorization": f"Bearer {LS_KEY}", "Accept": "application/json"},
                             params=params, timeout=45)
            r.raise_for_status(); return r.json()
        except (requests.exceptions.ReadTimeout, requests.exceptions.ConnectionError):
            if a == 3: raise
            time.sleep(3 * 2 ** a)

aa_resp = aa_get("/data/llms/models")
aa_data = aa_resp.get("data", aa_resp) if isinstance(aa_resp, dict) else aa_resp

ls_models, cursor = [], None
while True:
    d = ls_get("/models", {"cursor": cursor} if cursor else None)
    ls_models += d.get("models", [])
    cursor = d.get("next_cursor")
    if not cursor:
        break
    time.sleep(1)

print(f"AA: {len(aa_data)} models | LLM Stats: {len(ls_models)} models")

In [ ]:
# --- merge (same field mapping as build_models_json.main) ---
ls_lookup = build_ls_lookup(ls_models)
models = []
for m in aa_data:
    ev = m.get("evaluations") or {}
    pr = m.get("pricing") or {}
    ls = enrich_from_ls(m["name"], ls_lookup)
    org = m["model_creator"]["name"]
    models.append({
        "id": m["id"], "name": m["name"], "slug": m.get("slug", ""),
        "org": org, "country": org_country(org), "release_date": m.get("release_date"),
        "open_weight": ls["open_weight"], "param_count": ls["param_count"],
        "license": ls["license"], "modalities": ls["modalities"],
        "intelligence_index": ev.get("artificial_analysis_intelligence_index"),
        "coding_index": ev.get("artificial_analysis_coding_index"),
        "math_index": ev.get("artificial_analysis_math_index"),
        "gpqa": ev.get("gpqa"), "hle": ev.get("hle"), "mmlu_pro": ev.get("mmlu_pro"),
        "livecodebench": ev.get("livecodebench"), "ifbench": ev.get("ifbench"),
        "lcr": ev.get("lcr"), "aime_25": ev.get("aime_25"),
        "price_input": pr.get("price_1m_input_tokens"),
        "price_output": pr.get("price_1m_output_tokens"),
        "price_blended": pr.get("price_1m_blended_3_to_1"),
        "tokens_per_sec": m.get("median_output_tokens_per_second"),
        "ttft": m.get("median_time_to_first_token_seconds"),
    })
print(f"merged {len(models)} models | open={sum(1 for x in models if x['open_weight'] is True)}")

In [ ]:
# --- write Delta ---
from pyspark.sql import functions as F
from pyspark.sql.types import (StructType, StructField, StringType, BooleanType,
                                DoubleType, LongType, ArrayType)

schema = StructType([
    StructField("id", StringType()), StructField("name", StringType()),
    StructField("slug", StringType()), StructField("org", StringType()),
    StructField("country", StringType()), StructField("release_date", StringType()),
    StructField("open_weight", BooleanType()), StructField("param_count", LongType()),
    StructField("license", StringType()), StructField("modalities", ArrayType(StringType())),
    StructField("intelligence_index", DoubleType()), StructField("coding_index", DoubleType()),
    StructField("math_index", DoubleType()), StructField("gpqa", DoubleType()),
    StructField("hle", DoubleType()), StructField("mmlu_pro", DoubleType()),
    StructField("livecodebench", DoubleType()), StructField("ifbench", DoubleType()),
    StructField("lcr", DoubleType()), StructField("aime_25", DoubleType()),
    StructField("price_input", DoubleType()), StructField("price_output", DoubleType()),
    StructField("price_blended", DoubleType()), StructField("tokens_per_sec", DoubleType()),
    StructField("ttft", DoubleType()),
])

df = (spark.createDataFrame(models, schema=schema)
      .withColumn("release_date", F.to_date("release_date"))
      .withColumn("captured_at", F.current_date()))

# 1) current snapshot — always overwrite
(df.write.mode("overwrite").option("overwriteSchema", "true")
   .saveAsTable(f"{CATALOG}.{SCHEMA}.model_benchmarks"))
print("model_benchmarks rows:", spark.table(f"{CATALOG}.{SCHEMA}.model_benchmarks").count())

# 2) history — append only if >= HISTORY_EVERY_DAYS since the last snapshot
hist = f"{CATALOG}.{SCHEMA}.model_benchmarks_history"
row = spark.sql(f"SELECT max(captured_at) AS last, datediff(current_date(), max(captured_at)) AS days_since FROM {hist}").collect()[0]
last, days_since = row["last"], row["days_since"]
if last is None or days_since >= HISTORY_EVERY_DAYS:
    df.write.mode("append").saveAsTable(hist)
    print(f"appended history snapshot (previous: {last}, {days_since}d ago)")
else:
    print(f"history current (last {last}, {days_since}d ago) — next append in {HISTORY_EVERY_DAYS - days_since}d")
print("history rows:", spark.table(hist).count())
display(spark.table(f"{CATALOG}.{SCHEMA}.model_benchmarks").orderBy(F.col("intelligence_index").desc_nulls_last()).limit(5))

## Scheduling
Run this as **one daily Job** (Workflows → task = this notebook, **Source = Git provider / branch `main`**, serverless).
- `model_benchmarks` refreshes every day.
- `model_benchmarks_history` self-paces: it appends only when `today − max(captured_at) ≥ HISTORY_EVERY_DAYS` (14), so you get a clean biweekly trail from one daily schedule — no separate biweekly job.
- The job's run-as identity needs **READ** on the `frontier_labs` secret scope.